# Step 2: Feature Engineering

以 `reels_shortcode` 為 key，將各來源資料拼成兩張特徵表：
- `new_features.csv`：bass 224 筆 + static_info、embedding、ads_type
- `hist_features.csv`：hist_reels 10487 筆 + embedding、ads_type

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. 讀取原始資料

In [3]:
hist_reels = pd.read_csv(RAW / 'hist_reels.csv')
bass       = pd.read_csv(RAW / 'bass_parameters_by_reel.csv')
static     = pd.read_csv(RAW / 'reels_static_info.csv')
ads        = pd.read_csv(RAW / 'ads_type_results.csv')
ads        = ads.rename(columns={'Reels代號': 'reels_shortcode'})

# embedding：只取指定欄位
emb_cols = ['Reels連結', 'KOL帳號', 'best_topic_labels'] + \
           [f'emb_{i:03d}' for i in range(32)]
embedding = pd.read_csv(RAW / 'reels_embedding.csv', usecols=emb_cols,
                        engine='python', on_bad_lines='skip')

# 從 URL 抽出 shortcode
embedding['reels_shortcode'] = (
    embedding['Reels連結'].str.extract(r'/reel/([^/]+)/')
)

print('hist_reels :', hist_reels.shape)
print('bass       :', bass.shape)
print('static     :', static.shape)
print('ads        :', ads.shape)
print('embedding  :', embedding.shape)

hist_reels : (10487, 11)
bass       : (224, 8)
static     : (791, 5)
ads        : (3124, 16)
embedding  : (10569, 36)


In [4]:
print(bass['reels_shortcode'].head(5))
print(embedding['reels_shortcode'].head(5))
print(len(set(bass['reels_shortcode']).intersection(set(embedding['reels_shortcode']))))

0    DW79gHugehL
1    DW7-Fy2gDyV
2    DW8FPlIzLWC
3    DW79Sg0D74o
4    DW8OGwwkynP
Name: reels_shortcode, dtype: str
0    DPeT4BHExo-
1    DPZEm_oExnd
2    DLvRZSlTBzn
3    DIvNtToTBL-
4    Ciksu1-vDBo
Name: reels_shortcode, dtype: str
13


## 2. 建立 new_features（bass 224 筆）

In [5]:
new_features = (
    bass
    .merge(static,    on='reels_shortcode', how='left')
    .merge(embedding, on='reels_shortcode', how='left')
    .merge(ads,       on='reels_shortcode', how='left')
)

print('new_features shape :', new_features.shape)
print('\n欄位：')
print(new_features.columns.tolist())

new_features shape : (226, 62)

欄位：
['reels_shortcode', 'p_first4d', 'q_first4d', 'M_first4d', 'p_after4d', 'q_after4d', 'M_after4d', 'constant_after4d_time0_views', 'kol_account', 'post_time', 'duration', 'caption', 'KOL帳號_x', 'Reels連結_x', 'best_topic_labels', 'emb_000', 'emb_001', 'emb_002', 'emb_003', 'emb_004', 'emb_005', 'emb_006', 'emb_007', 'emb_008', 'emb_009', 'emb_010', 'emb_011', 'emb_012', 'emb_013', 'emb_014', 'emb_015', 'emb_016', 'emb_017', 'emb_018', 'emb_019', 'emb_020', 'emb_021', 'emb_022', 'emb_023', 'emb_024', 'emb_025', 'emb_026', 'emb_027', 'emb_028', 'emb_029', 'emb_030', 'emb_031', 'KOL帳號_y', '資料抓取時間', '發文時間', '影片長度(秒)', 'Reels連結_y', '觀看數', '讚數', '留言數', '文案長度', '前10則熱門留言', '完整文案', 'AI業配判定', '品牌名稱', '產品類型', '業配類型']


## 3. 建立 hist_features（hist_reels 10487 筆）

In [6]:
hist_features = (
    hist_reels
    .merge(embedding, on='reels_shortcode', how='left')
    .merge(ads,       on='reels_shortcode', how='left')
)

print('hist_features shape :', hist_features.shape)
print('\n欄位：')
print(hist_features.columns.tolist())

hist_features shape : (11309, 61)

欄位：
['kol_account', 'reels_shortcode', 'post_time', 'duration', 'views', 'plays', 'likes', 'comments', 'is_promo', 'timestamp', 'caption', 'KOL帳號_x', 'Reels連結_x', 'best_topic_labels', 'emb_000', 'emb_001', 'emb_002', 'emb_003', 'emb_004', 'emb_005', 'emb_006', 'emb_007', 'emb_008', 'emb_009', 'emb_010', 'emb_011', 'emb_012', 'emb_013', 'emb_014', 'emb_015', 'emb_016', 'emb_017', 'emb_018', 'emb_019', 'emb_020', 'emb_021', 'emb_022', 'emb_023', 'emb_024', 'emb_025', 'emb_026', 'emb_027', 'emb_028', 'emb_029', 'emb_030', 'emb_031', 'KOL帳號_y', '資料抓取時間', '發文時間', '影片長度(秒)', 'Reels連結_y', '觀看數', '讚數', '留言數', '文案長度', '前10則熱門留言', '完整文案', 'AI業配判定', '品牌名稱', '產品類型', '業配類型']


In [7]:
print('new_features shape :', new_features.shape)
print('hist_features shape:', hist_features.shape)

print('\nbass reels_shortcode sample:')
print(bass['reels_shortcode'].head(5).tolist())

print('\nembedding reels_shortcode sample:')
print(embedding['reels_shortcode'].head(5).tolist())

print('\nads reels_shortcode sample:')
print(ads['reels_shortcode'].head(5).tolist())

new_features shape : (226, 62)
hist_features shape: (11309, 61)

bass reels_shortcode sample:
['DW79gHugehL', 'DW7-Fy2gDyV', 'DW8FPlIzLWC', 'DW79Sg0D74o', 'DW8OGwwkynP']

embedding reels_shortcode sample:
['DPeT4BHExo-', 'DPZEm_oExnd', 'DLvRZSlTBzn', 'DIvNtToTBL-', 'Ciksu1-vDBo']

ads reels_shortcode sample:
['CvyjbvoJBLG', 'CjxK2_vDyd4', 'CwchdjvhOSx', 'DRjB6izCZY9', 'DNe0hEoBT-4']


## 4. 檢查 merge 遺失情況

In [8]:
print('=== new_features 各欄位缺值數 ===')
print(new_features.isnull().sum()[new_features.isnull().sum() > 0])

print('\n=== hist_features 各欄位缺值數 ===')
print(hist_features.isnull().sum()[hist_features.isnull().sum() > 0])

=== new_features 各欄位缺值數 ===
p_first4d              1
q_first4d              1
M_first4d              1
KOL帳號_x              213
Reels連結_x            213
best_topic_labels    213
emb_000              213
emb_001              213
emb_002              213
emb_003              213
emb_004              213
emb_005              213
emb_006              213
emb_007              213
emb_008              213
emb_009              213
emb_010              213
emb_011              213
emb_012              213
emb_013              213
emb_014              213
emb_015              213
emb_016              213
emb_017              213
emb_018              213
emb_019              213
emb_020              213
emb_021              213
emb_022              213
emb_023              213
emb_024              213
emb_025              213
emb_026              213
emb_027              213
emb_028              213
emb_029              213
emb_030              213
emb_031              213
KOL帳號_y              2

In [9]:
for df_name, df in [('new_features', new_features), ('hist_features', hist_features)]:
    df.drop(columns=['KOL帳號_y', 'Reels連結_y'], errors='ignore', inplace=True)
    df.rename(columns={'KOL帳號_x': 'KOL帳號', 'Reels連結_x': 'Reels連結'}, inplace=True)
    df.drop_duplicates(subset='reels_shortcode', keep='first', inplace=True)

print('new_features shape :', new_features.shape)
print('hist_features shape:', hist_features.shape)

new_features shape : (223, 60)
hist_features shape: (10373, 59)


In [10]:
print(new_features['best_topic_labels'].isna().sum())
print(new_features['AI業配判定'].value_counts(dropna=False))
print(new_features['emb_000'].isna().sum())

210
AI業配判定
NaN     219
True      4
Name: count, dtype: int64
210


In [11]:
import numpy as np

# 讀取 1024 維新 embedding
raw = np.load(RAW / 'reel_embeddings_with_code_new.npy', allow_pickle=True)

emb_new_cols = [f'emb_new_{i:03d}' for i in range(1024)]
emb_new_df = pd.DataFrame({
    'reels_shortcode': [row['reels_code'] for row in raw],
    **{col: [row['embedding'][i] for row in raw] for i, col in enumerate(emb_new_cols)}
})

print('emb_new_df shape:', emb_new_df.shape)

# Merge 進 new_features 和 hist_features
new_features  = new_features.merge(emb_new_df, on='reels_shortcode', how='left')
hist_features = hist_features.merge(emb_new_df, on='reels_shortcode', how='left')

print('new_features  shape:', new_features.shape,  '  emb_new_000 缺值:', new_features['emb_new_000'].isna().sum())
print('hist_features shape:', hist_features.shape, '  emb_new_000 缺值:', hist_features['emb_new_000'].isna().sum())

emb_new_df shape: (780, 1025)
new_features  shape: (224, 1084)   emb_new_000 缺值: 0
hist_features shape: (10375, 1083)   emb_new_000 缺值: 10083


## 5. 存檔

In [12]:
new_features.to_csv(PROCESSED / 'new_features.csv', index=False)
hist_features.to_csv(PROCESSED / 'hist_features.csv', index=False)

print('已儲存：')
print(f'  {PROCESSED / "new_features.csv"}')
print(f'  {PROCESSED / "hist_features.csv"}')

已儲存：
  ../data/processed/new_features.csv
  ../data/processed/hist_features.csv
